# Tutorial 1: Simple Medical Image Segmentation on SageMaker

This notebook demonstrates how to train a medical image segmentation model on Amazon SageMaker using a single GPU.

## What You'll Learn
- Setting up a SageMaker training job
- Training a SegResNet model for 3D medical image segmentation
- Deploying the model to a SageMaker endpoint
- Running inference on medical images
- Cleaning up resources

## Prerequisites
- Medical imaging data in S3 (organized as train/valid/test folders)
- SageMaker execution role with S3 access

## Step 1: Setup and Imports

In [ ]:
import os
import sagemaker
from sagemaker.pytorch import PyTorch, PyTorchModel
import boto3


sagemaker_session = sagemaker.Session(boto3.Session(region_name='us-east-1'))
# You can use a region like us-east-1.x


## Step 1b: Create Sagemaker policy

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '../..'))
from utils import get_or_create_role

role = get_or_create_role()
print(f"SageMaker role: {role}")


In [ ]:
# IF the role is giving an error, uncomment the following line and provide your role ARN
# The role should have necessary permissions for SageMaker operations AmazonSageMakerFullAccess and AmazonS3FullAccess
# role = "arn:aws:iam::123456789012:role/service-role/AmazonSageMaker-ExecutionRole-20200101T000001"

region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()
print(f"SageMaker role: {role}")
print(f"Region: {region}")
print(f"Bucket: {bucket}")

## Step 2: Configure Data Paths

In [6]:
bucket = 'public-datasets-imaging-us-east-1'  # Replace with your actual bucket name
data_path = f's3://{bucket}/segmentation_data/'
output_path = f's3://{bucket}/segmentation_data/output'

print(f"Training data: {data_path}")
print(f"Output path: {output_path}")

Training data: s3://public-datasets-imaging-us-east-1/segmentation_data/
Output path: s3://public-datasets-imaging-us-east-1/segmentation_data/output


## Step 3: Define Hyperparameters

In [7]:
hyperparameters = {
    "model_name": "SegResNet",
    "batch_size": 2,
    "epochs": 2,
    "val_interval": 2,
    "lr": 1e-4
}

print("Hyperparameters:")
for key, value in hyperparameters.items():
    print(f"  {key}: {value}")

Hyperparameters:
  model_name: SegResNet
  batch_size: 2
  epochs: 2
  val_interval: 2
  lr: 0.0001


## Step 4: Create SageMaker Estimator

In [8]:
estimator = PyTorch(
    entry_point="train_simple.py",
    source_dir="../code/training",
    role=role,
    instance_count=1,
    instance_type="ml.g5.xlarge",
    framework_version="2.1.0",
    py_version="py310",
    hyperparameters=hyperparameters,
    output_path=output_path,
    base_job_name="medical-seg-simple",
    keep_alive_period_in_seconds=1800,
    environment={"PIP_CACHE_DIR": "/opt/ml/sagemaker/warmpoolcache/pip"},
    sagemaker_session=sagemaker_session,
)
print("✓ Estimator created successfully!")

✓ Estimator created successfully!


## Step 5: Start Training

In [9]:
estimator.fit({"training": data_path}, wait=True, logs="All")

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: medical-seg-simple-2026-06-12-15-47-25-720


Using provided s3_resource
2026-06-12 15:47:26 Starting - Starting the training job...
2026-06-12 15:47:26 Pending - Training job waiting for capacity.........
2026-06-12 15:49:22 Pending - Preparing the instances for training...
2026-06-12 15:49:56 Downloading - Downloading input data......
2026-06-12 15:50:36 Downloading - Downloading the training image.....................
2026-06-12 15:54:28 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ci

## Step 6: View Training Results

In [10]:
model_data = estimator.model_data
training_job_name = estimator.latest_training_job.name
print(f"Training job: {training_job_name}")
print(f"Model artifacts: {model_data}")

Training job: medical-seg-simple-2026-06-12-15-47-25-720
Model artifacts: s3://public-datasets-imaging-us-east-1/segmentation_data/output/medical-seg-simple-2026-06-12-15-47-25-720/output/model.tar.gz
